# Segmentação por Métodos Baseados em Regiões

Este notebook demonstra técnicas de **segmentação baseada em regiões** aplicadas a imagens de obras.

A ideia é dividir uma imagem em regiões com características semelhantes, como cor, intensidade, textura ou continuidade espacial.

Essa abordagem é útil para demonstrar conceitos de segmentação em visão computacional antes de avançar para modelos supervisionados, como YOLO, U-Net, DeepLab ou Mask R-CNN.

**Tema da aula:** Fluxo de um sistema baseado em visão computacional  
**Etapa:** Segmentação  
**Técnica:** Métodos Baseados em Regiões — *Region-Based Methods*  
**Aplicação:** imagens de obras, elementos estruturais, canteiros, superfícies e regiões de interesse  
**Versão:** 1.0 — Google Colab + Processamento em Lote

## 1. O que são métodos baseados em regiões?

Métodos baseados em regiões agrupam pixels que possuem características semelhantes.

Em vez de procurar apenas bordas, como Sobel ou Canny, esses métodos tentam responder:

> Quais partes da imagem pertencem a regiões visualmente homogêneas?

Uma região pode ser formada por pixels parecidos em cor, intensidade, textura, proximidade espacial, continuidade ou contraste em relação ao fundo.

Em imagens de obras, isso pode ajudar a separar visualmente estruturas, equipamentos, formas, paredes, lajes, áreas de solo, materiais ou regiões com alteração visual.

## 2. Diferença entre detecção de bordas e segmentação por regiões

A detecção de bordas procura transições bruscas:

```text
pixel claro → pixel escuro
```

A segmentação por regiões procura conjuntos de pixels semelhantes:

```text
região A → concreto
região B → céu
região C → equipamento
região D → solo
```

| Abordagem | Pergunta principal | Saída típica |
|---|---|---|
| Sobel / Canny | Onde estão as bordas? | linhas e contornos |
| Métodos por regiões | Quais pixels pertencem à mesma região? | máscaras e áreas segmentadas |
| Deep Learning | Que objeto ou classe aparece aqui? | classes, caixas, máscaras semânticas |

Neste notebook, o objetivo é didático: mostrar como a imagem pode ser dividida em regiões antes de usar modelos mais avançados.

## 3. Técnicas demonstradas

Serão demonstradas três abordagens:

### 1. K-means por cor

Agrupa pixels em `K` grupos de cores semelhantes.

### 2. Superpixels SLIC

Divide a imagem em pequenas regiões compactas e homogêneas.

### 3. Watershed

Trata a imagem como uma superfície topográfica e separa regiões por “bacias”.

Essas técnicas são clássicas e ajudam a entender o conceito de segmentação antes de aplicar redes neurais.

## 4. Atenção técnica

Esses métodos não fazem segmentação semântica.

Ou seja, eles não sabem que uma região é “pilar”, “forma”, “grua”, “parede” ou “trinca”.

Eles apenas dividem a imagem segundo critérios visuais.

Para reconhecimento semântico, normalmente usamos modelos supervisionados, como YOLO, U-Net, DeepLab ou Mask R-CNN.

## 5. Importação das bibliotecas

Execute esta célula para carregar as bibliotecas necessárias.

O notebook usa OpenCV, NumPy, Pandas, Matplotlib e scikit-image.

In [ ]:
import os
from pathlib import Path

import cv2 as cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

# Instalar/importar scikit-image, se necessário
try:
    from skimage.segmentation import slic, mark_boundaries
    from skimage.color import label2rgb
except ImportError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-image"])

    from skimage.segmentation import slic, mark_boundaries
    from skimage.color import label2rgb

## 6. Montagem do Google Drive e configuração das pastas

O notebook espera a seguinte estrutura no Google Drive:

```text
MyDrive/
└── Python_VC/
    ├── fotos_obra/
    └── output_segmentacao_regioes/
```

As imagens originais devem estar dentro de `fotos_obra`.

In [ ]:
# Montar Google Drive
drive.mount('/content/drive')

# Configurações principais
PASTA_BASE = '/content/drive/MyDrive/Python_VC'
PASTA_ENTRADA = os.path.join(PASTA_BASE, "fotos_obra")
PASTA_SAIDA = os.path.join(PASTA_BASE, "output_segmentacao_regioes")

# Extensões aceitas
EXTENSOES_IMAGEM = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp')

# Parâmetros de segmentação
PARAMS_SEGMENTACAO = {
    # Redimensionamento
    'redimensionar_max': 1200,

    # K-means
    'kmeans_k': 6,
    'kmeans_tentativas': 5,

    # SLIC superpixels
    'slic_n_segments': 180,
    'slic_compactness': 12,
    'slic_sigma': 1,

    # Watershed
    'watershed_gauss_ksize': 5,
    'watershed_inverter_threshold': True,
    'watershed_dist_threshold': 0.35,
    'watershed_kernel_morfologico': 3,

    # Relatórios e visualização
    'salvar_visualizacao': True
}

print("Configuração carregada.")
print(f"Pasta base: {PASTA_BASE}")
print(f"Pasta de entrada: {PASTA_ENTRADA}")
print(f"Pasta de saída: {PASTA_SAIDA}")
print(f"Parâmetros: {PARAMS_SEGMENTACAO}")

## 7. Verificação da configuração

In [ ]:
def verificar_configuracao():
    """Verifica se as pastas existem e se há imagens disponíveis."""
    print("🔍 VERIFICANDO CONFIGURAÇÃO")
    print(f"📁 Pasta base: {PASTA_BASE}")
    print(f"📂 Entrada: {PASTA_ENTRADA}")
    print(f"📂 Saída: {PASTA_SAIDA}")

    problemas = []

    if not os.path.exists(PASTA_BASE):
        problemas.append("❌ Pasta base não encontrada.")

    if not os.path.exists(PASTA_ENTRADA):
        problemas.append("❌ Pasta de entrada 'fotos_obra' não encontrada.")

    if os.path.exists(PASTA_ENTRADA):
        imagens = [
            f for f in os.listdir(PASTA_ENTRADA)
            if f.lower().endswith(EXTENSOES_IMAGEM)
        ]

        if not imagens:
            problemas.append("❌ Nenhuma imagem encontrada na pasta 'fotos_obra'.")
        else:
            print(f"📸 Imagens encontradas: {len(imagens)}")

    if problemas:
        print("\n".join(problemas))
        return False

    print("✅ Configuração verificada com sucesso.")
    return True


verificar_configuracao()

## 8. Funções auxiliares

In [ ]:
def criar_pasta(caminho):
    """Cria uma pasta, se ela ainda não existir."""
    os.makedirs(caminho, exist_ok=True)


def listar_imagens(pasta):
    """Lista imagens válidas na pasta."""
    return [
        f for f in os.listdir(pasta)
        if f.lower().endswith(EXTENSOES_IMAGEM)
    ]


def carregar_imagem(caminho):
    """Carrega uma imagem com OpenCV."""
    imagem = cv.imread(caminho)

    if imagem is None:
        raise ValueError(f"Não foi possível carregar a imagem: {caminho}")

    return imagem


def redimensionar_imagem(imagem, largura_max):
    """Redimensiona a imagem mantendo proporção."""
    h, w = imagem.shape[:2]

    if largura_max and largura_max > 0 and w > largura_max:
        proporcao = largura_max / w
        nova_largura = largura_max
        nova_altura = int(h * proporcao)

        imagem_redimensionada = cv.resize(
            imagem,
            (nova_largura, nova_altura),
            interpolation=cv.INTER_AREA
        )

        print(f"   📐 Redimensionada: {w}x{h} → {nova_largura}x{nova_altura}")

        return imagem_redimensionada

    return imagem


def salvar_imagem(caminho, imagem):
    """Salva uma imagem criando o diretório, se necessário."""
    criar_pasta(os.path.dirname(caminho))
    cv.imwrite(caminho, imagem)


def converter_bgr_para_rgb(imagem_bgr):
    """Converte imagem BGR para RGB."""
    return cv.cvtColor(imagem_bgr, cv.COLOR_BGR2RGB)


def converter_rgb_para_bgr(imagem_rgb):
    """Converte imagem RGB para BGR."""
    return cv.cvtColor(imagem_rgb, cv.COLOR_RGB2BGR)


def criar_mapa_colorido_rotulos(rotulos):
    """Cria uma imagem colorida a partir de um mapa de rótulos."""
    rotulos_int = rotulos.astype(np.int32)
    rotulos_unicos = np.unique(rotulos_int)

    rng = np.random.default_rng(42)
    max_label = int(rotulos_unicos.max()) if rotulos_unicos.size else 0
    cores = rng.integers(0, 255, size=(max_label + 1, 3), dtype=np.uint8)

    mapa_colorido = cores[rotulos_int]

    return mapa_colorido

## 9. Segmentação por K-means

O K-means agrupa os pixels em `K` grupos de cor.

Em imagens de obras, ele pode separar visualmente regiões como céu, solo, concreto, equipamentos, materiais, sombras e elementos com cores distintas.

O resultado depende muito do valor de `K`.

In [ ]:
def segmentar_kmeans(imagem_bgr, params):
    """Segmenta a imagem por agrupamento K-means em espaço RGB."""
    imagem_rgb = converter_bgr_para_rgb(imagem_bgr)

    pixels = imagem_rgb.reshape((-1, 3)).astype(np.float32)

    k = params['kmeans_k']

    criterios = (
        cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER,
        100,
        0.2
    )

    _, labels, centros = cv.kmeans(
        pixels,
        k,
        None,
        criterios,
        params['kmeans_tentativas'],
        cv.KMEANS_PP_CENTERS
    )

    centros = np.uint8(centros)

    imagem_segmentada_rgb = centros[labels.flatten()].reshape(imagem_rgb.shape)

    labels_img = labels.reshape(imagem_rgb.shape[:2])

    mapa_colorido = criar_mapa_colorido_rotulos(labels_img)

    return labels_img, imagem_segmentada_rgb, mapa_colorido

## 10. Segmentação por Superpixels SLIC

O SLIC cria **superpixels**, pequenas regiões compactas e visualmente homogêneas.

Essa técnica é muito didática porque mostra que a imagem pode ser dividida em regiões locais coerentes.

In [ ]:
def segmentar_slic(imagem_bgr, params):
    """Segmenta a imagem usando SLIC superpixels."""
    imagem_rgb = converter_bgr_para_rgb(imagem_bgr)

    segmentos = slic(
        imagem_rgb,
        n_segments=params['slic_n_segments'],
        compactness=params['slic_compactness'],
        sigma=params['slic_sigma'],
        start_label=1
    )

    bordas_slic = mark_boundaries(
        imagem_rgb,
        segmentos,
        color=(1, 1, 0)
    )

    bordas_slic_rgb = (bordas_slic * 255).astype(np.uint8)

    mapa_slic = label2rgb(
        segmentos,
        imagem_rgb,
        kind='avg',
        bg_label=0
    )

    mapa_slic_rgb = np.clip(mapa_slic, 0, 255).astype(np.uint8)

    return segmentos, bordas_slic_rgb, mapa_slic_rgb

## 11. Segmentação por Watershed

O Watershed interpreta a imagem como uma superfície topográfica.

Fluxo simplificado:

```text
Imagem em cinza
→ suavização
→ limiarização
→ morfologia
→ transformada de distância
→ marcadores
→ watershed
→ regiões segmentadas
```

Esse método pode separar objetos conectados, mas é sensível à qualidade dos marcadores.

In [ ]:
def segmentar_watershed(imagem_bgr, params):
    """Segmenta a imagem usando Watershed com marcadores automáticos."""
    imagem_processamento = imagem_bgr.copy()

    cinza = cv.cvtColor(imagem_processamento, cv.COLOR_BGR2GRAY)

    k_gauss = params['watershed_gauss_ksize']
    if k_gauss % 2 == 0:
        k_gauss += 1

    cinza_suave = cv.GaussianBlur(cinza, (k_gauss, k_gauss), 0)

    tipo_threshold = cv.THRESH_BINARY_INV if params['watershed_inverter_threshold'] else cv.THRESH_BINARY

    _, mascara = cv.threshold(
        cinza_suave,
        0,
        255,
        tipo_threshold + cv.THRESH_OTSU
    )

    k_morf = params['watershed_kernel_morfologico']
    if k_morf % 2 == 0:
        k_morf += 1

    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (k_morf, k_morf))

    abertura = cv.morphologyEx(mascara, cv.MORPH_OPEN, kernel, iterations=1)
    fechamento = cv.morphologyEx(abertura, cv.MORPH_CLOSE, kernel, iterations=2)

    fundo_provavel = cv.dilate(fechamento, kernel, iterations=3)

    dist_transform = cv.distanceTransform(fechamento, cv.DIST_L2, 5)

    _, primeiro_plano = cv.threshold(
        dist_transform,
        params['watershed_dist_threshold'] * dist_transform.max(),
        255,
        0
    )

    primeiro_plano = np.uint8(primeiro_plano)

    desconhecido = cv.subtract(fundo_provavel, primeiro_plano)

    _, marcadores = cv.connectedComponents(primeiro_plano)

    marcadores = marcadores + 1
    marcadores[desconhecido == 255] = 0

    labels_watershed = cv.watershed(imagem_processamento, marcadores)

    labels_para_cor = labels_watershed.copy()
    labels_para_cor[labels_para_cor < 0] = 0

    mapa_watershed_colorido = criar_mapa_colorido_rotulos(labels_para_cor)

    bordas_overlay = imagem_bgr.copy()
    bordas_overlay[labels_watershed == -1] = (0, 0, 255)

    return labels_watershed, mapa_watershed_colorido, bordas_overlay, fechamento

## 12. Processamento completo de uma imagem

Esta função aplica os três métodos na mesma imagem:

```text
Imagem original
→ K-means
→ SLIC
→ Watershed
→ comparativo
→ métricas
```

In [ ]:
def processar_imagem_segmentacao(nome_arquivo):
    """Processa uma única imagem com métodos baseados em regiões."""
    caminho_imagem = os.path.join(PASTA_ENTRADA, nome_arquivo)

    print(f"🔄 Processando segmentação por regiões: {nome_arquivo}")

    imagem_bgr = carregar_imagem(caminho_imagem)

    imagem_bgr = redimensionar_imagem(
        imagem_bgr,
        PARAMS_SEGMENTACAO['redimensionar_max']
    )

    imagem_rgb = converter_bgr_para_rgb(imagem_bgr)

    labels_kmeans, kmeans_segmentada_rgb, kmeans_colorida = segmentar_kmeans(
        imagem_bgr,
        PARAMS_SEGMENTACAO
    )

    labels_slic, slic_bordas_rgb, slic_mapa_rgb = segmentar_slic(
        imagem_bgr,
        PARAMS_SEGMENTACAO
    )

    labels_watershed, watershed_colorido, watershed_overlay_bgr, mascara_watershed = segmentar_watershed(
        imagem_bgr,
        PARAMS_SEGMENTACAO
    )

    nome_base = os.path.splitext(nome_arquivo)[0]
    pasta_imagem = os.path.join(PASTA_SAIDA, nome_base)
    criar_pasta(pasta_imagem)

    caminhos = {
        'original': os.path.join(pasta_imagem, f"{nome_base}_original.png"),
        'kmeans_segmentada': os.path.join(pasta_imagem, f"{nome_base}_kmeans_segmentada.png"),
        'kmeans_regioes': os.path.join(pasta_imagem, f"{nome_base}_kmeans_regioes.png"),
        'slic_bordas': os.path.join(pasta_imagem, f"{nome_base}_slic_bordas.png"),
        'slic_regioes': os.path.join(pasta_imagem, f"{nome_base}_slic_regioes.png"),
        'watershed_mascara': os.path.join(pasta_imagem, f"{nome_base}_watershed_mascara.png"),
        'watershed_regioes': os.path.join(pasta_imagem, f"{nome_base}_watershed_regioes.png"),
        'watershed_overlay': os.path.join(pasta_imagem, f"{nome_base}_watershed_overlay.png"),
        'comparativo': os.path.join(pasta_imagem, f"{nome_base}_comparativo_segmentacao_regioes.png")
    }

    salvar_imagem(caminhos['original'], imagem_bgr)
    salvar_imagem(caminhos['kmeans_segmentada'], converter_rgb_para_bgr(kmeans_segmentada_rgb))
    salvar_imagem(caminhos['kmeans_regioes'], converter_rgb_para_bgr(kmeans_colorida))
    salvar_imagem(caminhos['slic_bordas'], converter_rgb_para_bgr(slic_bordas_rgb))
    salvar_imagem(caminhos['slic_regioes'], converter_rgb_para_bgr(slic_mapa_rgb))
    salvar_imagem(caminhos['watershed_mascara'], mascara_watershed)
    salvar_imagem(caminhos['watershed_regioes'], converter_rgb_para_bgr(watershed_colorido))
    salvar_imagem(caminhos['watershed_overlay'], watershed_overlay_bgr)

    if PARAMS_SEGMENTACAO['salvar_visualizacao']:
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))

        axes[0, 0].imshow(imagem_rgb)
        axes[0, 0].set_title("Original")
        axes[0, 0].axis("off")

        axes[0, 1].imshow(kmeans_segmentada_rgb)
        axes[0, 1].set_title(f"K-means segmentado | K={PARAMS_SEGMENTACAO['kmeans_k']}")
        axes[0, 1].axis("off")

        axes[0, 2].imshow(kmeans_colorida)
        axes[0, 2].set_title("K-means | regiões coloridas")
        axes[0, 2].axis("off")

        axes[1, 0].imshow(slic_bordas_rgb)
        axes[1, 0].set_title(f"SLIC superpixels | n≈{PARAMS_SEGMENTACAO['slic_n_segments']}")
        axes[1, 0].axis("off")

        axes[1, 1].imshow(slic_mapa_rgb)
        axes[1, 1].set_title("SLIC | regiões médias")
        axes[1, 1].axis("off")

        axes[1, 2].imshow(converter_bgr_para_rgb(watershed_overlay_bgr))
        axes[1, 2].set_title("Watershed | bordas")
        axes[1, 2].axis("off")

        plt.suptitle(f"Segmentação baseada em regiões — {nome_arquivo}", fontsize=16)
        plt.tight_layout()
        plt.savefig(caminhos['comparativo'], dpi=150, bbox_inches="tight")
        plt.close()

    n_regioes_kmeans = int(len(np.unique(labels_kmeans)))
    n_regioes_slic = int(len(np.unique(labels_slic)))
    labels_watershed_validos = labels_watershed[labels_watershed > 1]
    n_regioes_watershed = int(len(np.unique(labels_watershed_validos))) if labels_watershed_validos.size > 0 else 0

    resultado = {
        'imagem': nome_arquivo,
        'largura': imagem_bgr.shape[1],
        'altura': imagem_bgr.shape[0],
        'kmeans_k': PARAMS_SEGMENTACAO['kmeans_k'],
        'regioes_kmeans': n_regioes_kmeans,
        'slic_n_segments_param': PARAMS_SEGMENTACAO['slic_n_segments'],
        'regioes_slic_geradas': n_regioes_slic,
        'regioes_watershed': n_regioes_watershed,
        'caminho_original': caminhos['original'],
        'caminho_kmeans_segmentada': caminhos['kmeans_segmentada'],
        'caminho_kmeans_regioes': caminhos['kmeans_regioes'],
        'caminho_slic_bordas': caminhos['slic_bordas'],
        'caminho_slic_regioes': caminhos['slic_regioes'],
        'caminho_watershed_overlay': caminhos['watershed_overlay'],
        'caminho_comparativo': caminhos['comparativo']
    }

    print(
        f"✅ Concluído: {nome_arquivo} | "
        f"K-means: {n_regioes_kmeans} regiões | "
        f"SLIC: {n_regioes_slic} regiões | "
        f"Watershed: {n_regioes_watershed} regiões"
    )

    return resultado

## 13. Processamento em lote

Esta função aplica os métodos de segmentação em todas as imagens da pasta `fotos_obra`.

In [ ]:
def processar_lote_segmentacao_regioes():
    """Processa todas as imagens da pasta de entrada."""
    print("🚀 INICIANDO SEGMENTAÇÃO POR MÉTODOS BASEADOS EM REGIÕES")
    print("=" * 70)

    if not verificar_configuracao():
        print("\n❌ Não foi possível iniciar o processamento.")
        return []

    criar_pasta(PASTA_SAIDA)

    imagens = listar_imagens(PASTA_ENTRADA)

    print(f"📊 Total de imagens para processar: {len(imagens)}")
    print(f"📁 Resultados serão salvos em: {PASTA_SAIDA}")
    print("=" * 70)

    resultados = []
    erros = 0

    for i, nome_arquivo in enumerate(imagens, 1):
        print(f"\n[{i}/{len(imagens)}]")

        try:
            resultado = processar_imagem_segmentacao(nome_arquivo)
            resultados.append(resultado)

        except Exception as e:
            print(f"❌ Erro ao processar {nome_arquivo}: {str(e)}")
            erros += 1

    print("\n" + "=" * 70)
    print("📈 RESUMO DO PROCESSAMENTO")
    print(f"✅ Imagens processadas com sucesso: {len(resultados)}")
    print(f"❌ Erros: {erros}")

    if resultados:
        media_slic = np.mean([r['regioes_slic_geradas'] for r in resultados])
        media_watershed = np.mean([r['regioes_watershed'] for r in resultados])

        print(f"📌 Média de regiões SLIC: {media_slic:.1f}")
        print(f"📌 Média de regiões Watershed: {media_watershed:.1f}")

    return resultados

## 14. Relatório CSV e resumo TXT

In [ ]:
def gerar_relatorios_segmentacao(resultados):
    """Gera relatório CSV e resumo TXT."""
    if not resultados:
        print("ℹ️ Não há resultados para gerar relatório.")
        return None

    df = pd.DataFrame(resultados)

    caminho_csv = os.path.join(PASTA_SAIDA, "relatorio_segmentacao_regioes.csv")
    df.to_csv(caminho_csv, index=False)

    caminho_txt = os.path.join(PASTA_SAIDA, "resumo_segmentacao_regioes.txt")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write("RELATÓRIO DE SEGMENTAÇÃO POR MÉTODOS BASEADOS EM REGIÕES\n")
        f.write("=" * 70 + "\n")
        f.write(f"Total de imagens processadas: {len(resultados)}\n")
        f.write(f"Parâmetros utilizados: {PARAMS_SEGMENTACAO}\n")

        media_slic = np.mean([r['regioes_slic_geradas'] for r in resultados])
        media_watershed = np.mean([r['regioes_watershed'] for r in resultados])

        f.write(f"Regiões K-means por imagem: {PARAMS_SEGMENTACAO['kmeans_k']}\n")
        f.write(f"Média de regiões SLIC geradas: {media_slic:.1f}\n")
        f.write(f"Média de regiões Watershed geradas: {media_watershed:.1f}\n")

        f.write("\nDETALHES POR IMAGEM:\n")

        for r in resultados:
            f.write(
                f"- {r['imagem']}: "
                f"K-means={r['regioes_kmeans']}, "
                f"SLIC={r['regioes_slic_geradas']}, "
                f"Watershed={r['regioes_watershed']}\n"
            )

    print(f"📄 Relatório CSV salvo em: {caminho_csv}")
    print(f"📋 Resumo TXT salvo em: {caminho_txt}")

    return df

## 15. Visualização de exemplos

In [ ]:
def mostrar_exemplos_segmentacao(num_exemplos=3):
    """Mostra exemplos comparativos salvos na pasta de saída."""
    if not os.path.exists(PASTA_SAIDA):
        print("ℹ️ Pasta de saída ainda não existe.")
        return

    comparativos = []

    for raiz, _, arquivos in os.walk(PASTA_SAIDA):
        for arquivo in arquivos:
            if arquivo.endswith("_comparativo_segmentacao_regioes.png"):
                comparativos.append(os.path.join(raiz, arquivo))

    if not comparativos:
        print("ℹ️ Nenhum comparativo encontrado.")
        return

    exemplos = comparativos[:num_exemplos]

    print(f"🖼️ Exibindo {len(exemplos)} exemplo(s):")

    for caminho in exemplos:
        img = plt.imread(caminho)

        plt.figure(figsize=(16, 9))
        plt.imshow(img)
        plt.title(os.path.basename(caminho))
        plt.axis("off")
        plt.tight_layout()
        plt.show()

## 16. Execução completa

Execute esta célula para rodar todo o fluxo:

1. verificar configuração;
2. segmentar imagens com K-means;
3. segmentar imagens com SLIC;
4. segmentar imagens com Watershed;
5. salvar comparativos;
6. gerar relatório;
7. exibir exemplos.

In [ ]:
resultados_segmentacao = processar_lote_segmentacao_regioes()

df_segmentacao = gerar_relatorios_segmentacao(resultados_segmentacao)

mostrar_exemplos_segmentacao(num_exemplos=3)

## 17. Visualizar tabela do relatório

In [ ]:
if 'df_segmentacao' in globals() and isinstance(df_segmentacao, pd.DataFrame):
    display(df_segmentacao)
else:
    print("Execute primeiro a célula de processamento completo.")

## 18. Teste rápido com uma única imagem

Antes de processar todas as imagens, você pode testar os parâmetros com apenas uma imagem.

In [ ]:
imagens_disponiveis = listar_imagens(PASTA_ENTRADA) if os.path.exists(PASTA_ENTRADA) else []

if imagens_disponiveis:
    print(f"Imagem de teste: {imagens_disponiveis[0]}")
    _ = processar_imagem_segmentacao(imagens_disponiveis[0])
    mostrar_exemplos_segmentacao(num_exemplos=1)
else:
    print("Nenhuma imagem disponível para teste.")

## 19. Como ajustar os parâmetros

### K-means

Menos grupos de cor:

```python
PARAMS_SEGMENTACAO['kmeans_k'] = 4
```

Mais detalhes:

```python
PARAMS_SEGMENTACAO['kmeans_k'] = 10
```

### SLIC

Regiões maiores:

```python
PARAMS_SEGMENTACAO['slic_n_segments'] = 80
```

Regiões menores e mais detalhadas:

```python
PARAMS_SEGMENTACAO['slic_n_segments'] = 300
```

### Watershed

Separar menos regiões:

```python
PARAMS_SEGMENTACAO['watershed_dist_threshold'] = 0.50
```

Separar mais regiões:

```python
PARAMS_SEGMENTACAO['watershed_dist_threshold'] = 0.25
```

Se o objeto de interesse for claro em fundo escuro:

```python
PARAMS_SEGMENTACAO['watershed_inverter_threshold'] = False
```

In [ ]:
# Ajustes opcionais de parâmetros

# PARAMS_SEGMENTACAO['kmeans_k'] = 8
# PARAMS_SEGMENTACAO['slic_n_segments'] = 250
# PARAMS_SEGMENTACAO['slic_compactness'] = 15
# PARAMS_SEGMENTACAO['watershed_dist_threshold'] = 0.40
# PARAMS_SEGMENTACAO['watershed_inverter_threshold'] = True

PARAMS_SEGMENTACAO

## 20. Como interpretar os resultados

### K-means

Bom para demonstrar separação por cor.

Limitação: ignora semântica e pode misturar objetos diferentes com cores parecidas.

### SLIC

Bom para mostrar regiões pequenas e coerentes.

Limitação: ainda não sabe o que cada região representa.

### Watershed

Bom para separar objetos e regiões conectadas.

Limitação: depende muito da qualidade dos marcadores e pode supersegmentar.

Em uma aula, o mais importante é mostrar que segmentar não é apenas detectar bordas. Segmentar significa atribuir pixels a regiões.

## 21. Relação com modelos avançados

Os métodos baseados em regiões são importantes para entender a lógica da segmentação.

Em aplicações mais avançadas, podemos passar de:

```text
regiões visuais não supervisionadas
```

para:

```text
máscaras semânticas supervisionadas
```

Exemplo:

```text
K-means / SLIC / Watershed
→ conceitos de regiões
→ anotação de máscaras
→ treinamento de U-Net ou DeepLab
→ segmentação semântica de elementos da obra
```

## 22. Fluxo recomendado em visão computacional para obras

Um fluxo completo poderia ser:

```text
Imagem da obra
→ pré-processamento
→ remoção de ruído
→ realce de contraste
→ segmentação baseada em regiões
→ análise de regiões candidatas
→ modelo supervisionado
→ validação técnica
→ relatório ou dashboard
```

No contexto do slide da aula, este notebook corresponde à etapa:

```text
03 — Segmentação
→ Métodos Baseados em Regiões
```